# IGNITE shot designer

Design actuator trajectories for a DIII-D shot and simulate them with IGNITE.

- **Kernel:** *FusionAIHub (stellarai)*. If it is missing, install it once with
  `pixi run --frozen -e stellarai python -m ipykernel install --user --name fusionaihub-stellarai`.
- **Reasoning:** an AI agent does it. In Claude Code in this repository the designer is an MCP
  server (`.mcp.json`, server `ignite-shot-designer`), so you can ask e.g.
  *"Reproduce shot 204011 but with reactor-like rotation after 3.5 s and keeping CER"*.
  The agent saves its plan as JSON; load it in tab **4 plan**.
- **This notebook:** the same tools by hand.
  1. Look at a shot.
  2. Find precedents in all tokenized shots.
  3. Enumerate beam sets for a torque target.
  4. Write or load a plan, check it and plot it.
  5. Export it and simulate it with IGNITE (one GPU job, only after you tick the approval box).
     A run rolls out every seed twice, under the measured actuators (replay) and under the
     plan, with the same random stream, so the two differ only by the actuators.

Plan op reference: `tools.machine()["knowledge"]`, section *Plan ops*.

In [ ]:
%matplotlib inline
from tokamak_foundation_model.ignite.designer import gui, tools

gui.app(204011)

## The same steps as code

The choices below (which sources inject counter-current, which must stay on) are the
designer's, i.e. yours or the agent's; the tools only measure and check them against the
database. Here: zero net beam torque from 3.5 s in 204011 with NB21L/NB21R counter-current
and NB30L/NB33L kept on. `beam_options` lists every set of other sources to switch off; for a
set that needs more counter power than the database shows was ever run, it gives the largest
fraction of the kept sources' power at which it stays within the database.

In [ ]:
opts = tools.beam_options(204011, 3.5, counter=["NB21L", "NB21R"],
                          keep_on=["NB30L", "NB33L"], target_Nm=0.0)
print(opts["n_feasible"], "feasible of", opts["n_options"])
best = opts["options"][0]
best

In [ ]:
plan = {"shot": 204011,
        "goal": "zero net beam torque from 3.5 s",
        "rationale": "NB21L/R counter-current, NB30L/NB33L kept (power reduced where needed)",
        "ops": best["ops"] if best["feasible"] else best["at_max_fraction"]["ops"]}
check = tools.check_plan(plan)
print(check["ok"]); check["findings"]

In [ ]:
from tokamak_foundation_model.ignite.designer import plan as pl
gui.plot_actuators(pl.build(plan));

In [ ]:
# Writes the IGNITE input; prints the sbatch command. Nothing is submitted.
exported = tools.export_plan(plan, name="example_204011_zerotorque")
exported

In [ ]:
# GPU rollout (about 10-20 min per seed). Only with your approval:
# tools.simulate(exported["run_dir"], seeds=[1, 2, 3, 4], confirm=True)
# tools.simulation_status(exported["run_dir"])
# gui.plot_results(exported["run_dir"]); tools.simulation_results(exported["run_dir"])